## FD_abs

In [1]:
import sys
from google.colab import drive
drive.mount('/content/drive')
folder_path = '/content/drive/MyDrive/MCX_data'
sys.path.append(folder_path)

Mounted at /content/drive


In [2]:
import glob
import os
import pickle
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np
import sys
! pip install pmcx
from google.colab import drive
drive.mount('/content/drive')
folder_path = '/content/drive/MyDrive/MCX_data'
sys.path.append(folder_path)
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import numpy as np
import pickle
from sklearn.preprocessing import StandardScaler

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 127.3 MB/s eta 0:00:00
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Read the saved CSV

In [3]:
csv_path = "/content/drive/MyDrive/MCX_data/result_folder/training_fd_110MHz.csv"
df = pd.read_csv(csv_path)
distances = [10, 20, 30, 40]
metrics = ["uac", "udc", "phase_rad"]
sub = df[df["sds_key"].isin(distances)].copy()

wl_order = sorted(sub["wavelength_index"].unique())
ids = np.array(sorted(sub["simulation_id"].unique()))

feature_blocks = []
feature_names = []

for d in distances:
    for wl_idx in wl_order:
        tmp = (
            sub[(sub["sds_key"] == d) & (sub["wavelength_index"] == wl_idx)]
            .set_index("simulation_id")
            .loc[ids, metrics]
        )

        feature_blocks.append(tmp.to_numpy(dtype=np.float64))

        wl_label = f"wl{wl_idx + 1}"
        feature_names.extend([
            f"uac_d{d}_{wl_label}",
            f"udc_d{d}_{wl_label}",
            f"phase_d{d}_{wl_label}"
        ])

X = np.concatenate(feature_blocks, axis=1)

print(X.shape)
print(feature_names)

np.save("fd_features_Nx24.npy", X)
np.save("fd_features_ids.npy", ids)

out_df = pd.DataFrame(X, columns=feature_names)
training_set = out_df

(10000, 24)
['uac_d10_wl1', 'udc_d10_wl1', 'phase_d10_wl1', 'uac_d10_wl2', 'udc_d10_wl2', 'phase_d10_wl2', 'uac_d20_wl1', 'udc_d20_wl1', 'phase_d20_wl1', 'uac_d20_wl2', 'udc_d20_wl2', 'phase_d20_wl2', 'uac_d30_wl1', 'udc_d30_wl1', 'phase_d30_wl1', 'uac_d30_wl2', 'udc_d30_wl2', 'phase_d30_wl2', 'uac_d40_wl1', 'udc_d40_wl1', 'phase_d40_wl1', 'uac_d40_wl2', 'udc_d40_wl2', 'phase_d40_wl2']


In [11]:
csv_path = "/content/drive/MyDrive/MCX_data/result_folder/testing_fd_110MHz.csv"
df = pd.read_csv(csv_path)
distances = [10, 20, 30, 40]
metrics = ["uac", "udc", "phase_rad"]
sub = df[df["sds_key"].isin(distances)].copy()
wl_order = sorted(sub["wavelength_index"].unique())
ids = np.array(sorted(sub["simulation_id"].unique()))

feature_blocks = []
feature_names = []

for d in distances:
    for wl_idx in wl_order:
        tmp = (
            sub[(sub["sds_key"] == d) & (sub["wavelength_index"] == wl_idx)]
            .set_index("simulation_id")
            .loc[ids, metrics]
        )

        feature_blocks.append(tmp.to_numpy(dtype=np.float64))

        wl_label = f"wl{wl_idx + 1}"
        feature_names.extend([
            f"uac_d{d}_{wl_label}",
            f"udc_d{d}_{wl_label}",
            f"phase_d{d}_{wl_label}"
        ])

X = np.concatenate(feature_blocks, axis=1)

print(X.shape)
print(feature_names)

np.save("fd_features_Nx24.npy", X)
np.save("fd_features_ids.npy", ids)

out_df = pd.DataFrame(X, columns=feature_names)
testing_set = out_df

(1000, 24)
['uac_d10_wl1', 'udc_d10_wl1', 'phase_d10_wl1', 'uac_d10_wl2', 'udc_d10_wl2', 'phase_d10_wl2', 'uac_d20_wl1', 'udc_d20_wl1', 'phase_d20_wl1', 'uac_d20_wl2', 'udc_d20_wl2', 'phase_d20_wl2', 'uac_d30_wl1', 'udc_d30_wl1', 'phase_d30_wl1', 'uac_d30_wl2', 'udc_d30_wl2', 'phase_d30_wl2', 'uac_d40_wl1', 'udc_d40_wl1', 'phase_d40_wl1', 'uac_d40_wl2', 'udc_d40_wl2', 'phase_d40_wl2']


### GT

In [4]:
GT_folder_train = '/content/drive/MyDrive/MCX_data/stage2_csv/'
GT_folder_test = '/content/drive/MyDrive/MCX_data/test_csv/'

In [5]:
csv_files_train = glob.glob(os.path.join(GT_folder_train, '*.csv'))
GT_all_train = pd.concat([pd.read_csv(f) for f in csv_files_train], ignore_index=True)
csv_files_test = glob.glob(os.path.join(GT_folder_test, '*.csv'))
GT_all_test = pd.concat([pd.read_csv(f) for f in csv_files_test], ignore_index=True)

In [6]:
import numpy as np

sorted_ids = [i + 1 for i in range(10000)]

# ensure ID is int
GT_all_train["ID"] = GT_all_train["ID"].astype(int)

# filter + order by ID = 1..10000
GT_filtered = (GT_all_train[GT_all_train["ID"].isin(sorted_ids)]
               .copy()
               .set_index("ID")
               .loc[sorted_ids]
               .reset_index())

target_cols = ["HBO1","HHB1", "HBO2", "HHB2", "d1", "a1", "a2", "b1", "b2"]   # <-- adjust if needed

# (optional) verify all columns exist
missing = [c for c in target_cols if c not in GT_filtered.columns]
if missing:
    raise KeyError(f"Missing columns in GT_filtered: {missing}. Available: {list(GT_filtered.columns)}")

Y = GT_filtered[target_cols].to_numpy(dtype=np.float32)  # shape (N, 5)
Y_train = Y  # keep as (N,5) for multi-output regression

print("y_train shape:", Y_train.shape)
print("first row:", dict(zip(target_cols, Y_train[0])))

y_train shape: (10000, 9)
first row: {'HBO1': np.float32(10.618102), 'HHB1': np.float32(12.007143), 'HBO2': np.float32(46.95982), 'HHB2': np.float32(26.97317), 'd1': np.float32(12.0), 'a1': np.float32(1.8359671), 'a2': np.float32(1.2800592), 'b1': np.float32(2.1788228), 'b2': np.float32(2.103345)}


In [7]:
sorted_ids = [i + 1 for i in range(1000)]
GT_all_test["ID"] = GT_all_test["ID"].astype(int)
GT_filtered = (GT_all_test[GT_all_test["ID"].isin(sorted_ids)]
               .copy()
               .set_index("ID")
               .loc[sorted_ids]
               .reset_index())

target_cols = ["HBO1","HHB1", "HBO2", "HHB2", "d1", "a1", "a2", "b1", "b2"]   # <-- adjust if needed

# (optional) verify all columns exist
missing = [c for c in target_cols if c not in GT_filtered.columns]
if missing:
    raise KeyError(f"Missing columns in GT_filtered: {missing}. Available: {list(GT_filtered.columns)}")

Y = GT_filtered[target_cols].to_numpy(dtype=np.float32)  # shape (N, 5)
Y_test = Y  # keep as (N,5) for multi-output regression

print("y_test shape:", Y_test.shape)
print("first row:", dict(zip(target_cols, Y_test[0])))

y_test shape: (1000, 9)
first row: {'HBO1': np.float32(10.618102), 'HHB1': np.float32(12.007143), 'HBO2': np.float32(46.95982), 'HHB2': np.float32(26.97317), 'd1': np.float32(12.0), 'a1': np.float32(1.8359671), 'a2': np.float32(1.2800592), 'b1': np.float32(2.1788228), 'b2': np.float32(2.103345)}


In [12]:
X_train = training_set
X_test = testing_set

In [13]:
print(X_train.shape, X_test.shape, Y_train.shape, Y_test.shape)

(10000, 24) (1000, 24) (10000, 9) (1000, 9)


### Gaussian noise adding

In [14]:
def add_fdnirs_noise(X, acdc_rel, phase_std_rad, rng, clip_nonnegative=True):
    X = np.asarray(X, dtype=np.float64).copy()

    ac_idx = np.arange(0, X.shape[1], 3)
    dc_idx = np.arange(1, X.shape[1], 3)
    phase_idx = np.arange(2, X.shape[1], 3)

    # AC noise: relative Gaussian
    ac = X[:, ac_idx]
    ac_noise = rng.normal(loc=0.0, scale=np.abs(acdc_rel * ac), size=ac.shape)
    X[:, ac_idx] = ac + ac_noise

    # DC noise: relative Gaussian
    dc = X[:, dc_idx]
    dc_noise = rng.normal(loc=0.0, scale=np.abs(acdc_rel * dc), size=dc.shape)
    X[:, dc_idx] = dc + dc_noise

    # Phase noise: absolute Gaussian, in radians
    phase = X[:, phase_idx]
    phase_noise = rng.normal(loc=0.0, scale=phase_std_rad, size=phase.shape)
    X[:, phase_idx] = phase + phase_noise

    if clip_nonnegative:
        X[:, ac_idx] = np.clip(X[:, ac_idx], 0, None)
        X[:, dc_idx] = np.clip(X[:, dc_idx], 0, None)

    return X


# ============================================================
# Four noise levels
# AC/DC: relative noise
# Phase: literature values given in degrees -> convert to radians
# ============================================================

acdc_rel_levels = [0.0075, 0.015, 0.0375, 0.075]  # 1%, 2%, 5%, 10%

phase_std_deg_levels = [0.15, 0.3, 0.75, 1.5]   # degrees
phase_std_rad_levels = np.deg2rad(phase_std_deg_levels)  # radians

print("Phase std in degrees :", phase_std_deg_levels)
print("Phase std in radians :", phase_std_rad_levels)


# Use different RNGs so train/test noise are independent
rng_train = np.random.default_rng(123)
rng_test = np.random.default_rng(42)

# ============================================================
# Build noisy train/test sets
# ============================================================

noisy_trainsets = {}
noisy_testsets = {}

for level, (r, phase_std_rad) in enumerate(zip(acdc_rel_levels, phase_std_rad_levels), start=1):
    noisy_trainsets[f"level_{level}"] = add_fdnirs_noise(
        X=X_train,
        acdc_rel=r,
        phase_std_rad=phase_std_rad,
        rng=rng_train,
        clip_nonnegative=True,
    )

    noisy_testsets[f"level_{level}"] = add_fdnirs_noise(
        X=X_test,
        acdc_rel=r,
        phase_std_rad=phase_std_rad,
        rng=rng_test,
        clip_nonnegative=True,
    )

# ============================================================
# Unpack
# ============================================================

X_train_noise_1 = noisy_trainsets["level_1"]
X_train_noise_2 = noisy_trainsets["level_2"]
X_train_noise_3 = noisy_trainsets["level_3"]
X_train_noise_4 = noisy_trainsets["level_4"]

X_test_noise_1 = noisy_testsets["level_1"]
X_test_noise_2 = noisy_testsets["level_2"]
X_test_noise_3 = noisy_testsets["level_3"]
X_test_noise_4 = noisy_testsets["level_4"]

print("Train shape original :", X_train.shape)
print("Train noise 1 shape  :", X_train_noise_1.shape)
print("Train noise 2 shape  :", X_train_noise_2.shape)
print("Train noise 3 shape  :", X_train_noise_3.shape)
print("Train noise 4 shape  :", X_train_noise_4.shape)

print("Test shape original  :", X_test.shape)
print("Test noise 1 shape   :", X_test_noise_1.shape)
print("Test noise 2 shape   :", X_test_noise_2.shape)
print("Test noise 3 shape   :", X_test_noise_3.shape)
print("Test noise 4 shape   :", X_test_noise_4.shape)

Phase std in degrees : [0.15, 0.3, 0.75, 1.5]
Phase std in radians : [0.00261799 0.00523599 0.01308997 0.02617994]
Train shape original : (10000, 24)
Train noise 1 shape  : (10000, 24)
Train noise 2 shape  : (10000, 24)
Train noise 3 shape  : (10000, 24)
Train noise 4 shape  : (10000, 24)
Test shape original  : (1000, 24)
Test noise 1 shape   : (1000, 24)
Test noise 2 shape   : (1000, 24)
Test noise 3 shape   : (1000, 24)
Test noise 4 shape   : (1000, 24)


### KD LUT - Done!

In [ ]:
from sklearn.neighbors import KDTree


def fit_typewise_normalizer(X_train):
    X_train = np.asarray(X_train, dtype=np.float64)

    n_features = X_train.shape[1]
    if n_features % 3 != 0:
        raise ValueError("Expected number of features to be divisible by 3.")

    ac_idx = np.arange(0, n_features, 3)
    dc_idx = np.arange(1, n_features, 3)
    phase_idx = np.arange(2, n_features, 3)

    ac_mean = X_train[:, ac_idx].mean()
    ac_std  = X_train[:, ac_idx].std()
    dc_mean = X_train[:, dc_idx].mean()
    dc_std  = X_train[:, dc_idx].std()
    ph_mean = X_train[:, phase_idx].mean()
    ph_std  = X_train[:, phase_idx].std()

    eps = 1e-8
    ac_std = max(ac_std, eps)
    dc_std = max(dc_std, eps)
    ph_std = max(ph_std, eps)

    return {
        "ac_idx": ac_idx,
        "dc_idx": dc_idx,
        "phase_idx": phase_idx,
        "ac_mean": ac_mean,
        "ac_std": ac_std,
        "dc_mean": dc_mean,
        "dc_std": dc_std,
        "ph_mean": ph_mean,
        "ph_std": ph_std,
    }


def transform_typewise(X, stats):
    X = np.asarray(X, dtype=np.float64).copy()
    X[:, stats["ac_idx"]]    = (X[:, stats["ac_idx"]]    - stats["ac_mean"]) / stats["ac_std"]
    X[:, stats["dc_idx"]]    = (X[:, stats["dc_idx"]]    - stats["dc_mean"]) / stats["dc_std"]
    X[:, stats["phase_idx"]] = (X[:, stats["phase_idx"]] - stats["ph_mean"]) / stats["ph_std"]
    return X


# ============================================================
# 1) Build KDTree use X_train, no validation set is needed!
# ============================================================

def build_kd_lookup(X_train, Y_train, leaf_size=40):
    X_train = np.asarray(X_train, dtype=np.float64)
    Y_train = np.asarray(Y_train, dtype=np.float64)

    norm_stats = fit_typewise_normalizer(X_train)
    X_train_norm = transform_typewise(X_train, norm_stats)

    tree = KDTree(X_train_norm, leaf_size=leaf_size, metric="euclidean")

    model = {
        "tree": tree,
        "Y_train": Y_train,
        "norm_stats": norm_stats,
    }
    return model


def kd_lookup_predict_from_model(model, X_test, k=5, eps=1e-8):
    X_test = np.asarray(X_test, dtype=np.float64)
    X_test_norm = transform_typewise(X_test, model["norm_stats"])
    neighbor_dist, neighbor_idx = model["tree"].query(X_test_norm, k=k)  # find k neighbors from the data set!
    Y_neighbors = model["Y_train"][neighbor_idx]  # (n_test, k, n_targets)
    weights = 1.0 / (neighbor_dist + eps)

    # exact match handling
    zero_mask = neighbor_dist < eps
    has_zero = zero_mask.any(axis=1)
    if np.any(has_zero):
        weights[has_zero] = zero_mask[has_zero].astype(np.float64)

    weights = weights / weights.sum(axis=1, keepdims=True)
    Y_pred = np.sum(Y_neighbors * weights[:, :, None], axis=1)

    return Y_pred, neighbor_idx, neighbor_dist


# ============================================================
# 3) Metrics: overall + selected targets only
# ============================================================

def evaluate_selected_metrics(Y_true, Y_pred, target_numbers_1based=(3, 4), mape_eps=1e-8):
    Y_true = np.asarray(Y_true, dtype=np.float64)
    Y_pred = np.asarray(Y_pred, dtype=np.float64)

    if Y_true.ndim == 1:
        Y_true = Y_true.reshape(-1, 1)
    if Y_pred.ndim == 1:
        Y_pred = Y_pred.reshape(-1, 1)

    if Y_true.shape != Y_pred.shape:
        raise ValueError(f"Shape mismatch: Y_true {Y_true.shape}, Y_pred {Y_pred.shape}")

    target_idx = [t - 1 for t in target_numbers_1based]
    for idx in target_idx:
        if idx < 0 or idx >= Y_true.shape[1]:
            raise ValueError(f"Requested target index {idx} is out of range for Y with shape {Y_true.shape}")

    err = Y_pred - Y_true
    abs_err = np.abs(err)

    denom = np.maximum(np.abs(Y_true), mape_eps)
    ape = (abs_err / denom) * 100.0

    rows = []

    # overall: average across all targets per sample
    overall_mae_per_sample = abs_err.mean(axis=1)
    overall_mape_per_sample = ape.mean(axis=1)

    rows.append({
        "item": "overall",
        "MAE_mean": overall_mae_per_sample.mean(),
        "MAE_std": overall_mae_per_sample.std(),
        "MAPE_mean": overall_mape_per_sample.mean(),
        "MAPE_std": overall_mape_per_sample.std(),
    })

    # selected targets
    for tnum, idx in zip(target_numbers_1based, target_idx):
        rows.append({
            "item": f"target_{tnum}",
            "MAE_mean": abs_err[:, idx].mean(),
            "MAE_std": abs_err[:, idx].std(),
            "MAPE_mean": ape[:, idx].mean(),
            "MAPE_std": ape[:, idx].std(),
        })

    return rows


def print_metric_rows(pair_name, rows):
    print(f"\n===== {pair_name} =====")
    for r in rows:
        print(
            f"{r['item']:>10s} | "
            f"MAE = {r['MAE_mean']:.6f} ± {r['MAE_std']:.6f} | "
            f"MAPE = {r['MAPE_mean']:.6f} ± {r['MAPE_std']:.6f}"
        )


# ============================================================
# 4) Matched noisy train/test pairs
#    Assumes these already exist:
#    X_train_noise_1 ... X_train_noise_4
#    X_test_noise_1  ... X_test_noise_4
# ============================================================

train_test_pairs = {
    "noise_level_1": (X_train_noise_1, X_test_noise_1),
    "noise_level_2": (X_train_noise_2, X_test_noise_2),
    "noise_level_3": (X_train_noise_3, X_test_noise_3),
    "noise_level_4": (X_train_noise_4, X_test_noise_4),
}

# ============================================================
# 5) Run matched noisy-train / noisy-test evaluation
# ============================================================

k = 5
all_rows = []

for level_name, (X_train_cur, X_test_cur) in train_test_pairs.items():
    # IMPORTANT: rebuild KD-LUT for each noisy training set
    kd_model = build_kd_lookup(
        X_train=X_train_cur,
        Y_train=Y_train
    )

    Y_pred_cur, nn_idx_cur, nn_dist_cur = kd_lookup_predict_from_model(
        model=kd_model,
        X_test=X_test_cur,
        k=k
    )

    rows = evaluate_selected_metrics(
        Y_true=Y_test,
        Y_pred=Y_pred_cur,
        target_numbers_1based=(3, 4),
        mape_eps=1e-8
    )

    print_metric_rows(level_name, rows)

    for r in rows:
        all_rows.append({
            "noise_level": level_name,
            "trainset": level_name,
            "testset": level_name,
            **r
        })

# ============================================================
# 6) Summary table
# ============================================================

results_df = pd.DataFrame(all_rows)

print("\nSummary table:")
print(results_df)

summary_pivot = results_df.pivot(
    index="noise_level",
    columns="item",
    values=["MAE_mean", "MAE_std", "MAPE_mean", "MAPE_std"]
)

print("\nPivoted summary:")
print(summary_pivot)


===== noise_level_1 =====
   overall | MAE = 1.532654 ± 0.650339 | MAPE = 19.086939 ± 11.116903
  target_3 | MAE = 5.379815 ± 3.889054 | MAPE = 14.281095 ± 11.619142
  target_4 | MAE = 3.683738 ± 2.620753 | MAPE = 16.015054 ± 13.415180

===== noise_level_2 =====
   overall | MAE = 1.720129 ± 0.677842 | MAPE = 21.533515 ± 12.081462
  target_3 | MAE = 5.915141 ± 4.230767 | MAPE = 15.719909 ± 12.629269
  target_4 | MAE = 4.151671 ± 2.714189 | MAPE = 18.004675 ± 13.912331

===== noise_level_3 =====
   overall | MAE = 2.052634 ± 0.755946 | MAPE = 26.704323 ± 15.491549
  target_3 | MAE = 6.832182 ± 4.610906 | MAPE = 18.214063 ± 14.096520
  target_4 | MAE = 4.798395 ± 2.988352 | MAPE = 20.791719 ± 15.527387

===== noise_level_4 =====
   overall | MAE = 2.308494 ± 0.782382 | MAPE = 31.083223 ± 17.389961
  target_3 | MAE = 7.415378 ± 4.813669 | MAPE = 19.856081 ± 15.088266
  target_4 | MAE = 5.159727 ± 3.326046 | MAPE = 22.401522 ± 17.411442

Summary table:
      noise_level       trainset    

### ANN_done

In [20]:

def fit_typewise_normalizer(X_train):
    X_train = np.asarray(X_train, dtype=np.float64)

    n_features = X_train.shape[1]
    if n_features % 3 != 0:
        raise ValueError("Expected number of features to be divisible by 3.")

    ac_idx = np.arange(0, n_features, 3)
    dc_idx = np.arange(1, n_features, 3)
    phase_idx = np.arange(2, n_features, 3)

    ac_mean = X_train[:, ac_idx].mean()
    ac_std  = X_train[:, ac_idx].std()
    dc_mean = X_train[:, dc_idx].mean()
    dc_std  = X_train[:, dc_idx].std()
    ph_mean = X_train[:, phase_idx].mean()
    ph_std  = X_train[:, phase_idx].std()

    eps = 1e-8
    ac_std = max(ac_std, eps)
    dc_std = max(dc_std, eps)
    ph_std = max(ph_std, eps)

    return {
        "ac_idx": ac_idx,
        "dc_idx": dc_idx,
        "phase_idx": phase_idx,
        "ac_mean": ac_mean,
        "ac_std": ac_std,
        "dc_mean": dc_mean,
        "dc_std": dc_std,
        "ph_mean": ph_mean,
        "ph_std": ph_std,
    }


def transform_typewise(X, stats):
    X = np.asarray(X, dtype=np.float64).copy()
    X[:, stats["ac_idx"]]    = (X[:, stats["ac_idx"]]    - stats["ac_mean"]) / stats["ac_std"]
    X[:, stats["dc_idx"]]    = (X[:, stats["dc_idx"]]    - stats["dc_mean"]) / stats["dc_std"]
    X[:, stats["phase_idx"]] = (X[:, stats["phase_idx"]] - stats["ph_mean"]) / stats["ph_std"]
    return X

def evaluate_selected_metrics(Y_true, Y_pred, target_numbers_1based=(3, 4), mape_eps=1e-8):
    Y_true = np.asarray(Y_true, dtype=np.float64)
    Y_pred = np.asarray(Y_pred, dtype=np.float64)

    if Y_true.ndim == 1:
        Y_true = Y_true.reshape(-1, 1)
    if Y_pred.ndim == 1:
        Y_pred = Y_pred.reshape(-1, 1)

    if Y_true.shape != Y_pred.shape:
        raise ValueError(f"Shape mismatch: Y_true {Y_true.shape}, Y_pred {Y_pred.shape}")

    target_idx = [t - 1 for t in target_numbers_1based]
    for idx in target_idx:
        if idx < 0 or idx >= Y_true.shape[1]:
            raise ValueError(f"Requested target index {idx} is out of range for Y with shape {Y_true.shape}")

    err = Y_pred - Y_true
    abs_err = np.abs(err)

    denom = np.maximum(np.abs(Y_true), mape_eps)
    ape = (abs_err / denom) * 100.0

    rows = []

    # overall: average across all targets per sample
    overall_mae_per_sample = abs_err.mean(axis=1)
    overall_mape_per_sample = ape.mean(axis=1)

    rows.append({
        "item": "overall",
        "MAE_mean": overall_mae_per_sample.mean(),
        "MAE_std": overall_mae_per_sample.std(),
        "MAPE_mean": overall_mape_per_sample.mean(),
        "MAPE_std": overall_mape_per_sample.std(),
    })

    # selected targets
    for tnum, idx in zip(target_numbers_1based, target_idx):
        rows.append({
            "item": f"target_{tnum}",
            "MAE_mean": abs_err[:, idx].mean(),
            "MAE_std": abs_err[:, idx].std(),
            "MAPE_mean": ape[:, idx].mean(),
            "MAPE_std": ape[:, idx].std(),
        })

    return rows


def print_metric_rows(pair_name, rows):
    print(f"\n===== {pair_name} =====")
    for r in rows:
        print(
            f"{r['item']:>10s} | "
            f"MAE = {r['MAE_mean']:.6f} ± {r['MAE_std']:.6f} | "
            f"MAPE = {r['MAPE_mean']:.6f} ± {r['MAPE_std']:.6f}"
        )

In [18]:
def build_ann_model(n_features, n_targets, lr=1e-3, dropout=0.15):
    inputs = tf.keras.Input(shape=(n_features,))

    x = tf.keras.layers.Dense(256, activation="relu")(inputs)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(dropout)(x)

    x = tf.keras.layers.Dense(128, activation="relu")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(dropout)(x)

    x = tf.keras.layers.Dense(64, activation="relu")(x)
    x = tf.keras.layers.BatchNormalization()(x)

    outputs = tf.keras.layers.Dense(n_targets, activation="linear")(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="mse",
    )
    return model

In [21]:
import tensorflow as tf
from sklearn.model_selection import train_test_split


RANDOM_STATE = 42
MAPE_EPS = 1e-8

ANN_PARAMS = {
    "learning_rate": 0.15,
    "dropout": 0.10,
    "batch_size": 64,
    "val_fraction": 0.15,
    "max_epochs": 300,
}

tf.keras.utils.set_random_seed(RANDOM_STATE)


# ============================================================
# 1) Train one ANN for one noise level
# ============================================================

def train_ann_one_noise_level(
    X_train_cur,
    Y_train,
    learning_rate=0.15,
    dropout=0.10,
    batch_size=128,
    val_fraction=0.15,
    max_epochs=300,
    random_state=RANDOM_STATE,
):
    X_train_cur = np.asarray(X_train_cur, dtype=np.float32)
    Y_train = np.asarray(Y_train, dtype=np.float32)

    if Y_train.ndim == 1:
        Y_train = Y_train.reshape(-1, 1)

    # --------------------------------------------------------
    # Fit X normalizer on THIS noise-level training set only
    # --------------------------------------------------------

    X_tr, X_val, Y_tr, Y_val = train_test_split(
    X_train_cur,
    Y_train,
    test_size=val_fraction,
    random_state=random_state,
    shuffle=True,
    )

    norm_stats = fit_typewise_normalizer(X_tr)
    X_tr = transform_typewise(X_tr, norm_stats).astype(np.float32)
    X_val = transform_typewise(X_val, norm_stats).astype(np.float32)

    # --------------------------------------------------------
    # Build model
    # --------------------------------------------------------
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(random_state)

    model = build_ann_model(
        n_features=X_tr.shape[1],
        n_targets=Y_tr.shape[1],
        lr=learning_rate,
        dropout=dropout,
    )

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=20,
            restore_best_weights=True,
            verbose=1,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=6,
            min_lr=1e-6,
            verbose=1,
        ),
    ]

    history = model.fit(
        X_tr,
        Y_tr,
        validation_data=(X_val, Y_val),
        epochs=max_epochs,
        batch_size=batch_size,
        verbose=1,
        callbacks=callbacks,
    )

    best_epoch = int(np.argmin(history.history["val_loss"])) + 1

    # validation prediction in original scale
    Y_val_pred = model.predict(X_val, batch_size=batch_size, verbose=0)
    Y_val_pred = np.asarray(Y_val_pred, dtype=np.float64)

    return {
        "model": model,
        "history": history.history,
        "best_epoch": best_epoch,
        "norm_stats": norm_stats,
        "X_val_norm": X_val,
        "Y_val": np.asarray(Y_val, dtype=np.float64),
        "Y_val_pred": Y_val_pred,
        "params": {
            "learning_rate": learning_rate,
            "dropout": dropout,
            "batch_size": batch_size,
            "val_fraction": val_fraction,
            "max_epochs": max_epochs,
        },
    }


# ============================================================
# 2) Predict on one noisy test set using its own trained model
# ============================================================

def predict_ann_test_one_noise_level(train_bundle, X_test_cur):
    X_test_cur = np.asarray(X_test_cur, dtype=np.float32)

    X_test_norm = transform_typewise(
        X_test_cur,
        train_bundle["norm_stats"]
    ).astype(np.float32)

    Y_pred = train_bundle["model"].predict(
        X_test_norm,
        batch_size=train_bundle["params"]["batch_size"],
        verbose=0,
    )

    return np.asarray(Y_pred, dtype=np.float64)


# ============================================================
# 3) Train/validate/test separately for each noise level
# ============================================================

def run_ann_four_noise_levels(
    trainsets,
    testsets,
    Y_train,
    Y_test,
    learning_rate=0.01,
    dropout=0.10,
    batch_size=128,
    val_fraction=0.15,
    max_epochs=300,
    target_numbers_1based=(3, 4),
    mape_eps=1e-8,
    random_state=RANDOM_STATE,
):
    all_rows = []
    bundles = {}

    for noise_name in trainsets.keys():
        print("\n" + "=" * 80)
        print(f"Training ANN for {noise_name}")
        print("=" * 80)

        X_train_cur = trainsets[noise_name]
        X_test_cur = testsets[noise_name]

        # -------------------------
        # Train this noise-level ANN
        # -------------------------
        bundle = train_ann_one_noise_level(
            X_train_cur=X_train_cur,
            Y_train=Y_train,
            learning_rate=learning_rate,
            dropout=dropout,
            batch_size=batch_size,
            val_fraction=val_fraction,
            max_epochs=max_epochs,
            random_state=random_state,
        )
        bundles[noise_name] = bundle

        print(f"\n{noise_name} best_epoch: {bundle['best_epoch']}")
        print(f"{noise_name} params    : {bundle['params']}")

        # -------------------------
        # Validation metrics
        # -------------------------
        val_rows = evaluate_selected_metrics(
            Y_true=bundle["Y_val"],
            Y_pred=bundle["Y_val_pred"],
            target_numbers_1based=target_numbers_1based,
            mape_eps=mape_eps,
        )
        print_metric_rows(f"{noise_name} | validation_15pct | ann_fixed", val_rows)

        for r in val_rows:
            all_rows.append({
                "noise_level": noise_name,
                "split": "validation_15pct",
                "method": "ann_fixed",
                **r
            })

        # -------------------------
        # Test metrics
        # -------------------------
        Y_test_pred = predict_ann_test_one_noise_level(bundle, X_test_cur)

        test_rows = evaluate_selected_metrics(
            Y_true=Y_test,
            Y_pred=Y_test_pred,
            target_numbers_1based=target_numbers_1based,
            mape_eps=mape_eps,
        )
        print_metric_rows(f"{noise_name} | test | ann_fixed", test_rows)

        for r in test_rows:
            all_rows.append({
                "noise_level": noise_name,
                "split": "test",
                "method": "ann_fixed",
                **r
            })

        bundle["Y_test_pred"] = Y_test_pred

    results_df = pd.DataFrame(all_rows)
    return bundles, results_df


# ============================================================
# 4) Prepare noisy train/test sets for each level
# ============================================================

trainsets = {
    "noise_level_1": X_train_noise_1,
    "noise_level_2": X_train_noise_2,
    "noise_level_3": X_train_noise_3,
    "noise_level_4": X_train_noise_4,
}

testsets = {
    "noise_level_1": X_test_noise_1,
    "noise_level_2": X_test_noise_2,
    "noise_level_3": X_test_noise_3,
    "noise_level_4": X_test_noise_4,
}


# ============================================================
# 5) Run
# ============================================================

ann_bundles, ann_results_df = run_ann_four_noise_levels(
    trainsets=trainsets,
    testsets=testsets,
    Y_train=Y_train,
    Y_test=Y_test,
    learning_rate=ANN_PARAMS["learning_rate"],
    dropout=ANN_PARAMS["dropout"],
    batch_size=ANN_PARAMS["batch_size"],
    val_fraction=ANN_PARAMS["val_fraction"],   # 15%
    max_epochs=ANN_PARAMS["max_epochs"],
    target_numbers_1based=(3, 4),   # change to (2, 3) if needed
    mape_eps=MAPE_EPS,
    random_state=RANDOM_STATE,
)

print("\nFull summary table:")
print(ann_results_df)

# Optional pivot tables
summary_mae = ann_results_df.pivot_table(
    index=["noise_level", "split", "method"],
    columns="item",
    values=["MAE_mean", "MAE_std"],
)

summary_mape = ann_results_df.pivot_table(
    index=["noise_level", "split", "method"],
    columns="item",
    values=["MAPE_mean", "MAPE_std"],
)

print("\nMAE summary:")
print(summary_mae)

print("\nMAPE summary:")
print(summary_mape)

# Optional: only overall rows
overall_only = ann_results_df[ann_results_df["item"] == "overall"].copy()
overall_only = overall_only.sort_values(["noise_level", "split", "MAE_mean", "MAPE_mean"])

print("\nOverall-only summary:")
print(overall_only[["noise_level", "split", "method", "MAE_mean", "MAE_std", "MAPE_mean", "MAPE_std"]])


Training ANN for noise_level_1
Epoch 1/300
133/133 ━━━━━━━━━━━━━━━━━━━━ 10s 38ms/step - loss: 27.1264 - val_loss: 33.4311 - learning_rate: 0.1500
Epoch 2/300
133/133 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 15.9036 - val_loss: 39.5883 - learning_rate: 0.1500
Epoch 3/300
133/133 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 15.8185 - val_loss: 18.4650 - learning_rate: 0.1500
Epoch 4/300
133/133 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 15.2603 - val_loss: 18.2493 - learning_rate: 0.1500
Epoch 5/300
133/133 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 14.8340 - val_loss: 21.2926 - learning_rate: 0.1500
Epoch 6/300
133/133 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 14.7550 - val_loss: 67.5781 - learning_rate: 0.1500
Epoch 7/300
133/133 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 14.7502 - val_loss: 17.1824 - learning_rate: 0.1500
Epoch 8/300
133/133 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 14.6882 - val_loss: 19.8137 - learning_rate: 0.1500
Epoch 9/300
133/133 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1

### Different KNN methods

In [ ]:
import numpy as np
import pandas as pd
from sklearn.neighbors import (
    KDTree,
    NearestNeighbors,
    KNeighborsRegressor,
    RadiusNeighborsRegressor,
)

# ============================================================
# 1) Training-only normalization
# ============================================================

def fit_typewise_normalizer(X_train):
    X_train = np.asarray(X_train, dtype=np.float64)

    n_features = X_train.shape[1]
    if n_features % 3 != 0:
        raise ValueError("Expected number of features to be divisible by 3.")

    ac_idx = np.arange(0, n_features, 3)
    dc_idx = np.arange(1, n_features, 3)
    phase_idx = np.arange(2, n_features, 3)

    ac_mean = X_train[:, ac_idx].mean()
    ac_std  = X_train[:, ac_idx].std()
    dc_mean = X_train[:, dc_idx].mean()
    dc_std  = X_train[:, dc_idx].std()
    ph_mean = X_train[:, phase_idx].mean()
    ph_std  = X_train[:, phase_idx].std()

    eps = 1e-8
    ac_std = max(ac_std, eps)
    dc_std = max(dc_std, eps)
    ph_std = max(ph_std, eps)

    return {
        "ac_idx": ac_idx,
        "dc_idx": dc_idx,
        "phase_idx": phase_idx,
        "ac_mean": ac_mean,
        "ac_std": ac_std,
        "dc_mean": dc_mean,
        "dc_std": dc_std,
        "ph_mean": ph_mean,
        "ph_std": ph_std,
    }


def transform_typewise(X, stats):
    X = np.asarray(X, dtype=np.float64).copy()

    X[:, stats["ac_idx"]]    = (X[:, stats["ac_idx"]]    - stats["ac_mean"]) / stats["ac_std"]
    X[:, stats["dc_idx"]]    = (X[:, stats["dc_idx"]]    - stats["dc_mean"]) / stats["dc_std"]
    X[:, stats["phase_idx"]] = (X[:, stats["phase_idx"]] - stats["ph_mean"]) / stats["ph_std"]

    return X


# ============================================================
# 2) Helper utilities
# ============================================================

def _safe_raw_distance_weights(distances, power=1.0, eps=1e-8):
    distances = np.asarray(distances, dtype=np.float64)
    return 1.0 / np.power(np.maximum(distances, eps), power)


def make_inverse_square_weight_fn(eps=1e-8):
    def weight_fn(distances):
        return _safe_raw_distance_weights(distances, power=2.0, eps=eps)
    return weight_fn


def make_gaussian_weight_fn(sigma, eps=1e-8):
    sigma = float(max(sigma, eps))

    def weight_fn(distances):
        distances = np.asarray(distances, dtype=np.float64)
        return np.exp(-0.5 * (distances / sigma) ** 2)

    return weight_fn


def _normalize_manual_weights(distances, raw_weights, eps=1e-8):
    distances = np.asarray(distances, dtype=np.float64)
    raw_weights = np.asarray(raw_weights, dtype=np.float64)

    squeeze_back = False
    if distances.ndim == 1:
        distances = distances[None, :]
        raw_weights = raw_weights[None, :]
        squeeze_back = True

    zero_mask = distances < eps
    has_zero = zero_mask.any(axis=1)

    if np.any(has_zero):
        raw_weights[has_zero] = zero_mask[has_zero].astype(np.float64)

    row_sum = raw_weights.sum(axis=1, keepdims=True)
    bad = row_sum.squeeze(-1) <= 0

    if np.any(bad):
        raw_weights[bad] = 1.0
        row_sum = raw_weights.sum(axis=1, keepdims=True)

    weights = raw_weights / row_sum

    if squeeze_back:
        return weights[0]
    return weights


def estimate_neighbor_scale(X_train_norm, k=5):
    X_train_norm = np.asarray(X_train_norm, dtype=np.float64)
    n_train = len(X_train_norm)
    k_eff = min(k + 1, n_train)

    nbrs = NearestNeighbors(
        n_neighbors=k_eff,
        algorithm="kd_tree",
        metric="minkowski",
        p=2,
        n_jobs=-1,
    )
    nbrs.fit(X_train_norm)

    dists, _ = nbrs.kneighbors(X_train_norm)

    if dists.shape[1] >= 2:
        useful = dists[:, 1:]
    else:
        useful = dists

    positive = useful[useful > 0]
    if positive.size == 0:
        return 1.0, 1.0

    sigma = float(np.median(positive))
    radius = float(np.quantile(useful[:, -1], 0.95))
    sigma = max(sigma, 1e-8)
    radius = max(radius, 1e-8)
    return sigma, radius


# ============================================================
# 3) Exact KDTree Shepard baseline
# ============================================================

def kd_lookup_predict_exact(X_train_norm, Y_train, X_test_norm, k=5, eps=1e-8):
    X_train_norm = np.asarray(X_train_norm, dtype=np.float64)
    Y_train = np.asarray(Y_train, dtype=np.float64)
    X_test_norm = np.asarray(X_test_norm, dtype=np.float64)

    tree = KDTree(X_train_norm, leaf_size=40, metric="euclidean")
    neighbor_dist, neighbor_idx = tree.query(X_test_norm, k=k)

    Y_neighbors = Y_train[neighbor_idx]
    raw_weights = _safe_raw_distance_weights(neighbor_dist, power=1.0, eps=eps)
    weights = _normalize_manual_weights(neighbor_dist, raw_weights, eps=eps)

    Y_pred = np.sum(Y_neighbors * weights[:, :, None], axis=1)
    return Y_pred, neighbor_idx, neighbor_dist, tree


# ============================================================
# 4) Safe radius-neighbor prediction
# ============================================================

def predict_radius_regressor_safe(model, X_test_norm, Y_train, fallback_model, eps=1e-8):
    dists_list, idx_list = model.radius_neighbors(
        X_test_norm,
        return_distance=True,
        sort_results=True,
    )

    Y_train = np.asarray(Y_train, dtype=np.float64)
    fallback_pred = fallback_model.predict(X_test_norm)

    n_test = len(X_test_norm)
    n_targets = Y_train.shape[1]
    Y_pred = np.zeros((n_test, n_targets), dtype=np.float64)

    for i, (d, idx) in enumerate(zip(dists_list, idx_list)):
        if len(idx) == 0:
            Y_pred[i] = fallback_pred[i]
            continue

        y_neighbors = Y_train[idx]

        if model.weights in (None, "uniform"):
            raw_w = np.ones(len(idx), dtype=np.float64)
        elif model.weights == "distance":
            raw_w = _safe_raw_distance_weights(d, power=1.0, eps=eps)
        elif callable(model.weights):
            raw_w = np.asarray(model.weights(d), dtype=np.float64)
        else:
            raise ValueError(f"Unsupported radius weight mode: {model.weights}")

        w = _normalize_manual_weights(d, raw_w, eps=eps)
        Y_pred[i] = np.sum(y_neighbors * w[:, None], axis=0)

    return Y_pred


def print_metric_rows(header_name, rows):
    print(f"\n===== {header_name} =====")
    for r in rows:
        print(
            f"{r['item']:>10s} | "
            f"MAE = {r['MAE_mean']:.6f} ± {r['MAE_std']:.6f} | "
            f"MAPE = {r['MAPE_mean']:.6f} ± {r['MAPE_std']:.6f}"
        )


# ============================================================
# 6) Build all KNN-style models once
# ============================================================

def build_knn_models(X_train, Y_train, k=5, eps=1e-8):
    X_train = np.asarray(X_train, dtype=np.float64)
    Y_train = np.asarray(Y_train, dtype=np.float64)

    k = int(min(k, len(X_train)))
    if k < 1:
        raise ValueError("k must be >= 1.")

    norm_stats = fit_typewise_normalizer(X_train)
    X_train_norm = transform_typewise(X_train, norm_stats)

    sigma, radius = estimate_neighbor_scale(X_train_norm, k=k)
    gaussian_weight_fn = make_gaussian_weight_fn(sigma=sigma, eps=eps)
    inverse_square_weight_fn = make_inverse_square_weight_fn(eps=eps)

    official_models = {
        "knn_uniform_l2": KNeighborsRegressor(
            n_neighbors=k, weights="uniform", algorithm="kd_tree",
            metric="minkowski", p=2, leaf_size=40, n_jobs=-1
        ),

        "knn_inverse_square_l1": KNeighborsRegressor(
    n_neighbors=k, weights=inverse_square_weight_fn, algorithm="kd_tree",
    metric="minkowski", p=1, leaf_size=40, n_jobs=-1
),
        "knn_distance_l2": KNeighborsRegressor(
            n_neighbors=k, weights="distance", algorithm="kd_tree",
            metric="minkowski", p=2, leaf_size=40, n_jobs=-1
        ),
        "knn_uniform_l1": KNeighborsRegressor(
            n_neighbors=k, weights="uniform", algorithm="kd_tree",
            metric="minkowski", p=1, leaf_size=40, n_jobs=-1
        ),
        "knn_distance_l1": KNeighborsRegressor(
            n_neighbors=k, weights="distance", algorithm="kd_tree",
            metric="minkowski", p=1, leaf_size=40, n_jobs=-1
        ),
        "knn_inverse_square_l2": KNeighborsRegressor(
            n_neighbors=k, weights=inverse_square_weight_fn, algorithm="kd_tree",
            metric="minkowski", p=2, leaf_size=40, n_jobs=-1
        ),
        "knn_gaussian_l2": KNeighborsRegressor(
            n_neighbors=k, weights=gaussian_weight_fn, algorithm="kd_tree",
            metric="minkowski", p=2, leaf_size=40, n_jobs=-1
        ),
    }

    for model in official_models.values():
        model.fit(X_train_norm, Y_train)

    fallback_knn = KNeighborsRegressor(
        n_neighbors=k, weights="distance", algorithm="kd_tree",
        metric="minkowski", p=2, leaf_size=40, n_jobs=-1
    )
    fallback_knn.fit(X_train_norm, Y_train)

    radius_models = {
        "radius_uniform_l2": RadiusNeighborsRegressor(
            radius=radius, weights="uniform", algorithm="kd_tree",
            metric="minkowski", p=2, leaf_size=40, n_jobs=-1
        ),
        "radius_distance_l2": RadiusNeighborsRegressor(
            radius=radius, weights="distance", algorithm="kd_tree",
            metric="minkowski", p=2, leaf_size=40, n_jobs=-1
        ),
        "radius_gaussian_l2": RadiusNeighborsRegressor(
            radius=radius, weights=gaussian_weight_fn, algorithm="kd_tree",
            metric="minkowski", p=2, leaf_size=40, n_jobs=-1
        ),
    }

    for model in radius_models.values():
        model.fit(X_train_norm, Y_train)

    bundle = {
        "norm_stats": norm_stats,
        "X_train_norm": X_train_norm,
        "Y_train": Y_train,
        "k": k,
        "eps": eps,
        "sigma": sigma,
        "radius": radius,
        "official_models": official_models,
        "radius_models": radius_models,
        "fallback_knn": fallback_knn,
    }
    return bundle


# ============================================================
# 7) Predict one test set with all methods
# ============================================================

def predict_all_methods(model_bundle, X_test):
    X_test = np.asarray(X_test, dtype=np.float64)
    X_test_norm = transform_typewise(X_test, model_bundle["norm_stats"])

    results = {}

    # Official KNN regressors
    for name, model in model_bundle["official_models"].items():
        Y_pred = model.predict(X_test_norm)
        results[name] = Y_pred

    # Radius regressors
    for name, model in model_bundle["radius_models"].items():
        Y_pred = predict_radius_regressor_safe(
            model=model,
            X_test_norm=X_test_norm,
            Y_train=model_bundle["Y_train"],
            fallback_model=model_bundle["fallback_knn"],
            eps=model_bundle["eps"],
        )
        results[name] = Y_pred

    # Exact KDTree Shepard
    Y_pred_exact, nn_idx, nn_dist, kd_tree = kd_lookup_predict_exact(
        X_train_norm=model_bundle["X_train_norm"],
        Y_train=model_bundle["Y_train"],
        X_test_norm=X_test_norm,
        k=model_bundle["k"],
        eps=model_bundle["eps"],
    )
    results["kdtree_shepard_exact"] = Y_pred_exact

    return results


# ============================================================
# 8) Run all methods on all noisy test sets, LUT-style printing
# ============================================================

def run_knn_on_testsets(
    X_train, Y_train, Y_test, testsets, k=5, eps=1e-8,
    target_numbers_1based=(3, 4), mape_eps=1e-8
):
    model_bundle = build_knn_models(X_train=X_train, Y_train=Y_train, k=k, eps=eps)

    print("Normalization fitted on training only.")
    print("X_train_norm shape:", model_bundle["X_train_norm"].shape)
    print(f"Using k={model_bundle['k']}")
    print(f"Estimated Gaussian sigma: {model_bundle['sigma']:.6f}")
    print(f"Estimated radius         : {model_bundle['radius']:.6f}")

    all_rows = []

    for testset_name, X_test_cur in testsets.items():
        pred_dict = predict_all_methods(model_bundle, X_test_cur)

        for method_name, Y_pred_cur in pred_dict.items():
            rows = evaluate_selected_metrics(
                Y_true=Y_test,
                Y_pred=Y_pred_cur,
                target_numbers_1based=target_numbers_1based,
                mape_eps=mape_eps,
            )

            print_metric_rows(f"{testset_name} | {method_name}", rows)

            for r in rows:
                all_rows.append({
                    "testset": testset_name,
                    "method": method_name,
                    **r
                })

    results_df = pd.DataFrame(all_rows)
    return results_df


# ============================================================
# 9) Prepare your four noisy test sets
# ============================================================

testsets = {
    "noise_level_1": X_test_noise_1,
    "noise_level_2": X_test_noise_2,
    "noise_level_3": X_test_noise_3,
    "noise_level_4": X_test_noise_4,
}

# ============================================================
# 10) Run
# ============================================================

k = 5

results_df = run_knn_on_testsets(
    X_train=X_train,
    Y_train=Y_train,
    Y_test=Y_test,
    testsets=testsets,
    k=k,
    eps=1e-8,
    target_numbers_1based=(3, 4),   # change to (2, 3) if that is what you really want
    mape_eps=1e-8,
)

print("\nSummary table:")
print(results_df)

# Optional compact pivot tables
summary_mae = results_df.pivot_table(
    index=["testset", "method"],
    columns="item",
    values=["MAE_mean", "MAE_std"],
)

summary_mape = results_df.pivot_table(
    index=["testset", "method"],
    columns="item",
    values=["MAPE_mean", "MAPE_std"],
)

print("\nMAE summary:")
print(summary_mae)

print("\nMAPE summary:")
print(summary_mape)

# Optional: sort by overall MAE within each testset
overall_only = results_df[results_df["item"] == "overall"].copy()
overall_only = overall_only.sort_values(["testset", "MAE_mean", "MAPE_mean"])

print("\nSorted overall summary:")
print(overall_only[["testset", "method", "MAE_mean", "MAE_std", "MAPE_mean", "MAPE_std"]])

Normalization fitted on training only.
X_train_norm shape: (10000, 24)
Using k=5
Estimated Gaussian sigma: 0.132950
Estimated radius         : 0.281351

===== noise_level_1 | knn_uniform_l2 =====
   overall | MAE = 1.838357 ± 0.635579 | MAPE = 23.038409 ± 12.340837
  target_3 | MAE = 6.507633 ± 4.394902 | MAPE = 17.373081 ± 13.602977
  target_4 | MAE = 4.338471 ± 2.790186 | MAPE = 18.758775 ± 14.283851

===== noise_level_1 | knn_inverse_square_l1 =====
   overall | MAE = 0.913877 ± 0.640211 | MAPE = 11.372637 ± 9.853395
  target_3 | MAE = 3.284714 ± 3.103679 | MAPE = 8.754306 ± 9.132861
  target_4 | MAE = 2.272328 ± 2.117866 | MAPE = 9.906517 ± 10.341427

===== noise_level_1 | knn_distance_l2 =====
   overall | MAE = 1.456035 ± 0.663659 | MAPE = 18.177926 ± 11.309944
  target_3 | MAE = 5.176423 ± 3.924692 | MAPE = 13.774436 ± 11.782852
  target_4 | MAE = 3.465932 ± 2.513218 | MAPE = 15.043482 ± 12.730188

===== noise_level_1 | knn_uniform_l1 =====
   overall | MAE = 1.782668 ± 0.623633

In [ ]:
summary_mae

MAE_mean                       MAE_std  \
item                                  overall  target_3  target_4   overall   
testset       method                                                          
noise_level_1 kdtree_shepard_exact   1.456035  5.176423  3.465932  0.663659   
              knn_distance_l1        1.323942  4.752077  3.252917  0.585084   
              knn_distance_l2        1.456035  5.176423  3.465932  0.663659   
              knn_gaussian_l2        1.678675  5.955904  4.012325  0.635855   
              knn_inverse_square_l1  0.913877  3.284714  2.272328  0.640211   
              knn_inverse_square_l2  1.124398  4.009920  2.701761  0.766685   
              knn_uniform_l1         1.782668  6.380096  4.330500  0.623633   
              knn_uniform_l2         1.838357  6.507633  4.338471  0.635579   
              radius_distance_l2     2.011662  6.670515  4.528878  0.698812   
              radius_gaussian_l2     2.044530  6.844174  4.661297  0.685998   
              radius_uniform_l2      2.197605  7.251596  4.887796  0.685650   
noise_level_2 kdtree_shepard_exact   1.611931  5.625683  3.877876  0.660244   
              knn_distance_l1        1.486922  5.287404  3.598664  0.591232   
              knn_distance_l2        1.611931  5.625683  3.877876  0.660244   
              knn_gaussian_l2        1.702944  5.951644  4.112722  0.652973   
              knn_inverse_square_l1  1.195827  4.253648  2.919939  0.639748   
              knn_inverse_square_l2  1.381551  4.826245  3.337521  0.748323   
              knn_uniform_l1         1.771291  6.293330  4.244947  0.626804   
              knn_uniform_l2         1.848789  6.444824  4.424539  0.646841   
              radius_distance_l2     2.067918  6.863644  4.645445  0.698434   
              radius_gaussian_l2     2.045179  6.847065  4.648703  0.693336   
              radius_uniform_l2      2.190032  7.242271  4.865512  0.688820   
noise_level_3 kdtree_shepard_exact   1.895240  6.466199  4.447042  0.706393   
              knn_distance_l1        1.764003  6.126845  4.274343  0.644321   
              knn_distance_l2        1.895240  6.466199  4.447042  0.706393   
              knn_gaussian_l2        1.883544  6.441300  4.434777  0.727365   
              knn_inverse_square_l1  1.660726  5.759677  4.044211  0.685038   
              knn_inverse_square_l2  1.810108  6.162854  4.270803  0.759531   
              knn_uniform_l1         1.860044  6.466547  4.482825  0.642811   
              knn_uniform_l2         1.975574  6.748915  4.607825  0.690574   
              radius_distance_l2     2.114083  6.958026  4.744693  0.715519   
              radius_gaussian_l2     2.069546  6.845669  4.692234  0.719611   
              radius_uniform_l2      2.167873  7.120133  4.827471  0.709510   
noise_level_4 kdtree_shepard_exact   2.138393  7.087329  4.685514  0.773575   
              knn_distance_l1        2.047005  6.904985  4.652385  0.742162   
              knn_distance_l2        2.138393  7.087329  4.685514  0.773575   
              knn_gaussian_l2        2.110085  7.008485  4.634252  0.811323   
              knn_inverse_square_l1  2.015474  6.793611  4.584784  0.762445   
              knn_inverse_square_l2  2.109102  6.982461  4.626749  0.796636   
              knn_uniform_l1         2.077087  7.008940  4.718635  0.732818   
              knn_uniform_l2         2.166757  7.186474  4.745446  0.761009   
              radius_distance_l2     2.202370  7.188638  4.795103  0.767004   
              radius_gaussian_l2     2.169571  7.082718  4.746285  0.777723   
              radius_uniform_l2      2.225509  7.265832  4.831213  0.761761   

                                                         
item                                 target_3  target_4  
testset       method                                     
noise_level_1 kdtree_shepard_exact   3.924692  2.513218  
              knn_distance_l1        3.488783  2.297618  
              knn_distance_l2    